<a href="https://colab.research.google.com/github/Kongbeng-21/SuperAi/blob/main/601088_%E0%B8%81%E0%B8%A4%E0%B8%95%E0%B8%B4%E0%B8%98%E0%B8%B5_%E0%B8%89%E0%B8%B2%E0%B8%A2%E0%B9%81%E0%B8%AA%E0%B8%87.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/krittiteechaisang/dataset/sample_submission.csv
/kaggle/input/datasets/krittiteechaisang/dataset/questions.csv
/kaggle/input/datasets/krittiteechaisang/dataset/knowledge_base/products/KS-HP-008_kluensiang_gamestorm_h1.md
/kaggle/input/datasets/krittiteechaisang/dataset/knowledge_base/products/KS-SK-006_kluensiang_homepod_one.md
/kaggle/input/datasets/krittiteechaisang/dataset/knowledge_base/products/JC-HB-004_judchuam_dock_airbook_edition.md
/kaggle/input/datasets/krittiteechaisang/dataset/knowledge_base/products/KS-HP-004_kluensiang_headon_500.md
/kaggle/input/datasets/krittiteechaisang/dataset/knowledge_base/products/KS-EB-008_kluensiang_buds_z5_pro_gold_limited.md
/kaggle/input/datasets/krittiteechaisang/dataset/knowledge_base/products/SF-TB-007_saifah_tab_draw_pro.md
/kaggle/input/datasets/krittiteechaisang/dataset/knowledge_base/products/JC-CH-002_judchuam_charger_100w_usbc_dual.md
/kaggle/input/datasets/krittiteechaisang/dataset/knowledge_base/products/KS-E

# Setup

In [ ]:
!pip install openai faiss-cpu langchain langchain-community langchain-core langchain-text-splitters sentence-transformers tiktoken rank_bm25 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 22.8 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 71.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires r

In [ ]:
from kaggle_secrets import UserSecretsClient
TYPHOON_API_KEY = UserSecretsClient().get_secret("TYPHOON_API_KEY")

# Import and Config

In [ ]:
import os
import glob
import json
import re
import time
import pandas as pd
from openai import OpenAI

DATA_PATH = "/kaggle/input/datasets/krittiteechaisang/dataset"
KB_PATH   = "/kaggle/input/datasets/krittiteechaisang/dataset/knowledge_base"
OUTPUT_PATH      = "/kaggle/working/submission.csv"
CHECKPOINT_PATH  = "/kaggle/working/checkpoint.json"

MODEL_NAME           = "typhoon-v2.5-30b-a3b-instruct"
DELAY_BETWEEN_CALLS  = 1.0
MAX_RETRIES          = 3

client = OpenAI(
    api_key=TYPHOON_API_KEY,
    base_url="https://api.opentyphoon.ai/v1"
)
print("✓ Config เสร็จแล้ว")

✓ Config เสร็จแล้ว


# Load Knowledge Base

In [ ]:
import os
import glob
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from sentence_transformers import CrossEncoder # <-- นำเข้า Reranker
import numpy as np

# 1. โหลดและแบ่งข้อความ
documents = []
for filepath in glob.glob(os.path.join(KB_PATH, "**/*.md"), recursive=True):
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    documents.append(Document(page_content=content, metadata={"source": filepath}))

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100,
    separators=["\n\n", "\n", " ", ""]
)
chunks = text_splitter.split_documents(documents)
print(f"✓ แบ่งเอกสารได้ {len(chunks)} chunks")

# 2. สร้าง AI ค้นหา (Embeddings)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 10}) # ดึงมาเผื่อ 10 อัน

# สร้าง BM25 Search
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 10 # ดึงมาเผื่อ 10 อัน

# 3. โหลด Reranker Model (ตัวทำคะแนนความแม่นยำ)
print("กำลังโหลดโมเดล Reranker (อาจใช้เวลาสักครู่)...")
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3')
print("✓ โหลด Reranker เสร็จสิ้น")

def find_relevant_docs(question, choices, top_k=3):
    """ค้นหาแบบ Hybrid + Reranker """
    query = question + " " + " ".join(choices)

    # 1. ดึงข้อมูลหว่านๆ มาก่อนจากทั้ง Vector และ BM25
    try:
        vec_docs = vector_retriever.invoke(query)
        bm25_docs = bm25_retriever.invoke(query)
    except AttributeError:
        vec_docs = vector_retriever.get_relevant_documents(query)
        bm25_docs = bm25_retriever.get_relevant_documents(query)

    # รวมเอกสารและตัดตัวซ้ำ
    combined_content = list(set([doc.page_content for doc in vec_docs + bm25_docs]))

    if not combined_content:
        return ""

    # 2. ให้ Reranker ให้คะแนนความเป๊ะระหว่าง "คำถาม" กับ "เอกสารแต่ละชิ้น"
    pairs = [[question, doc] for doc in combined_content]
    scores = reranker.predict(pairs)

    # 3. เรียงลำดับจากคะแนนมากไปน้อย แล้วคัดมาแค่ top_k ที่ดีที่สุดจริงๆ
    ranked_indices = np.argsort(scores)[::-1]
    best_docs = [combined_content[i] for i in ranked_indices[:top_k]]

    return "\n\n===\n\n".join(best_docs)

print("✓ Hybrid + Reranker พร้อมใช้งานระดับ Grandmaster!")

✓ แบ่งเอกสารได้ 1396 chunks


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✓ โหลด Reranker เสร็จสิ้น
✓ Hybrid + Reranker พร้อมใช้งานระดับ Grandmaster!


# Prompt และ LLM

In [ ]:
import re
import time

def ask_typhoon(question, choices, context):
    choices_text = "\n".join(f"{i+1}. {c}" for i, c in enumerate(choices))

    # ใส่ Context เข้าไปให้ AI อ่าน
    prompt = f"""คุณคือ AI ผู้ช่วยตอบคำถามของร้านฟ้าใหม่ (ร้านขายอุปกรณ์อิเล็กทรอนิกส์)
จงตอบคำถามโดยอ้างอิงจากข้อมูลใน Context ด้านล่างนี้เป็นหลัก

Context ข้อมูลของร้าน:
{context}

กฎการตอบ:
- เลือกตัวเลือก 1-8: หากมีคำตอบที่ถูกต้องและอ้างอิงได้จาก Context
- เลือกตัวเลือก 9: หากใน Context ไม่มีข้อมูลที่สามารถตอบคำถามนี้ได้เลย
- เลือกตัวเลือก 10: หากคำถามไม่เกี่ยวกับร้านฟ้าใหม่เลย

คำถาม: {question}

ตัวเลือก:
{choices_text}

กรุณาวิเคราะห์และตอบเป็น "ตัวเลข 1-10" เพียงตัวเลขเดียวเท่านั้น ห้ามมีข้อความอื่นเด็ดขาด:"""

    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=4096  # รองรับ Prompt ยาวๆ
            )
            raw = resp.choices[0].message.content.strip()

            # ดึงเฉพาะตัวเลขจากคำตอบ
            for num_str in re.findall(r'\d+', raw):
                num = int(num_str)
                if 1 <= num <= 10:
                    return num
            return 9

        except Exception as e:
            err = str(e)
            if "429" in err or "quota" in err.lower():
                wait = 60 * (attempt + 1)
                print(f"  [RATE LIMIT] รอ {wait} วิ...")
                time.sleep(wait)
            else:
                print(f"  [ERROR] {e}")
                time.sleep(3)
    return 9

print("✓ ask_typhoon updated (แก้ max_tokens=4096)")

✓ ask_typhoon updated (แก้ max_tokens=4096)


# Checkpoint

In [ ]:
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print("✓ ลบ checkpoint แล้ว")
else:
    print("ไม่มี checkpoint")

ไม่มี checkpoint


In [ ]:
def load_checkpoint():
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH) as f:
            data = json.load(f)
        print(f"✓ โหลด checkpoint: {len(data)} rows บันทึกแล้ว")
        return data
    return {}

def save_checkpoint(results):
    with open(CHECKPOINT_PATH, "w") as f:
        json.dump(results, f, ensure_ascii=False)

def save_csv(results, total=100):
    rows = [{"id": i, "answer": results.get(str(i), 9)} for i in range(1, total + 1)]
    df = pd.DataFrame(rows)
    df.to_csv(OUTPUT_PATH, index=False)
    return df

print("✓ checkpoint functions พร้อมใช้งาน")

✓ checkpoint functions พร้อมใช้งาน


# Pipeline

In [ ]:
def run_pipeline():
    questions_df = pd.read_csv(os.path.join(DATA_PATH, "questions.csv"))
    total = len(questions_df)
    print(f"คำถามทั้งหมด: {total} ข้อ\n")

    results = load_checkpoint()
    already_done = set(results.keys())
    skipped = 0

    for _, row in questions_df.iterrows():
        qid      = str(row["id"])
        question = row["question"]
        choices  = [str(row[f"choice_{i}"]) for i in range(1, 11)]

        if qid in already_done:
            skipped += 1
            continue

        # ใช้ keyword search อย่างเดียว (ไม่ต้องใช้ vectorstore)
        context = find_relevant_docs(question, choices, top_k=5)

        answer = ask_typhoon(question, choices, context)

        if answer == 9:
            alt_context = find_relevant_docs(" ".join(choices), [], top_k=5)
            if alt_context:
                answer = ask_typhoon(question, choices, alt_context)

        results[qid] = answer

        status = "🔴" if answer in [9, 10] else "🟢"
        print(f"{status} Q{int(qid):3d}: ตอบ [{answer}] | {question[:55]}...")

        save_checkpoint(results)

        if int(qid) % 10 == 0:
            save_csv(results, total)
            print(f"บันทึก CSV ชั่วคราว ({qid} ข้อ)")

        time.sleep(DELAY_BETWEEN_CALLS)

    if skipped:
        print(f"\n(ข้ามไป {skipped} ข้อที่ทำแล้ว)")

    submission = save_csv(results, total)
    print(f"\n✓ บันทึกแล้ว: {OUTPUT_PATH}")
    print(f"   {len(submission)} rows")
    print(f"   answer=9: {(submission['answer']==9).sum()} ข้อ")
    print(f"   answer=10: {(submission['answer']==10).sum()} ข้อ")
    print(submission.head(10))
    return submission

print("✓ run_pipeline พร้อมใช้งาน")

✓ run_pipeline พร้อมใช้งาน


# Run

In [ ]:
submission = run_pipeline()

คำถามทั้งหมด: 100 ข้อ

✓ โหลด checkpoint: 4 rows บันทึกแล้ว
🟢 Q  5: ตอบ [6] | สั่งของจากฟ้าใหม่ สั่งได้ครั้งละกี่ชิ้นครับ...
🟢 Q  6: ตอบ [8] | ฟ้าใหม่จ่ายด้วย crypto ได้มั้ยคะ เช่น Bitcoin...
🟢 Q  7: ตอบ [1] | ซื้อ AirBook 14 มาแล้ว อยากรู้ว่าในกล่องมีอะไรมาบ้าง...
🟢 Q  8: ตอบ [1] | DuoPad สั่งซื้อได้เลยไหมครับ หรือต้องพรีออเดอร์...
🟢 Q  9: ตอบ [4] | เฮ้ยย อ่านเยอะมากเลย ขอถามนิดนึงนะคะ คือพอดีว่าจะไปเที่...
🟢 Q 10: ตอบ [7] | อยากใช้ซิม 2 ค่ายพร้อมกันครับ ใส่ซิมได้ 2 ใบมั้ย ไม่อยา...
บันทึก CSV ชั่วคราว (10 ข้อ)
🟢 Q 11: ตอบ [4] | โน้ตบุ๊คดาวเหนือ ครีเอเตอร์บุ๊ก 16 OLED ต่อจอเพิ่มได้กี...
🟢 Q 12: ตอบ [1] | จะซื้อโน้ตบุ๊คเล่น Genshin Impact ลื่นๆ กราฟิกสูงๆ มีรุ...
🟢 Q 13: ตอบ [3] | ทำงานในคาเฟ่เสียงดังมากเลย ต้องการหูฟังครอบหูที่ตัดเสีย...
🟢 Q 14: ตอบ [1] | ซื้อสายฟ้า X9 Pro มา อยากทราบว่าหัวชาร์จ 67W มาในกล่องด...
🟢 Q 15: ตอบ [7] | อยากฟังเพลง Lossless จาก streaming ได้คุณภาพดีๆ มีหูฟัง...
🟢 Q 16: ตอบ [1] | อยากได้นาฬิกาที่วัดคลื่นหัวใจ ECG ได้ มีรุ่นไหนบ้างคะ แ...
🟢 Q 17: ตอบ [8] | มอ

In [ ]:
import shutil
shutil.copy(OUTPUT_PATH, "/kaggle/working/submission_download.csv")
print("✓ พร้อม download แล้ว")